# 🚀 LLM in Python - From Simple to Complex

This notebook walks through practical Groq API examples, progressively building in complexity.

| # | Example | Difficulty |
|---|---------|------------|
| 1 | Setup & Basic Chat | ⭐ |
| 2 | System Prompts & Personas | ⭐⭐ |
| 3 | Streaming Responses | ⭐⭐ |
| 4 | Multi-turn Conversations | ⭐⭐⭐ |
| 5 | Structured JSON Output | ⭐⭐⭐ |
| 6 | Sentiment Analysis Pipeline | ⭐⭐⭐ |
| 7 | RAG — Retrieval-Augmented Generation | ⭐⭐⭐⭐ |
| 8 | Async Batch Processing | ⭐⭐⭐ |
| 9 | Prompt Chaining & Summarization | ⭐⭐⭐ |
| 10 | Interactive Chatbot UI (ipywidgets) | ⭐⭐⭐ |
| 11 | CSV / Data Analysis Agent | ⭐⭐⭐⭐ |

**> Get your free API key at:** https://console.groq.com/keys

## 🔧 Setup — Install & Configure

In [ ]:
!pip install groq faiss-cpu sentence-transformers ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.4 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

# Option A: Store your key in Colab Secrets (recommended)
# Go to the key icon in the left sidebar and add GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('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')
    print('Loaded API key from Colab Secrets')
except Exception:
    GROQ_API_KEY = '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'  # Replace with your key
    print('Using hardcoded key - prefer Colab Secrets for security')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

Using hardcoded key - prefer Colab Secrets for security


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    'fast':     'llama-3.1-8b-instant',
    'balanced': 'llama-3.3-70b-versatile',
    'code':     'qwen-2.5-coder-32b',
}

print('Groq client ready!')
print('Models:', list(MODELS.keys()))

Groq client ready!
Models: ['fast', 'balanced', 'code']


---
## Example 1 - Basic Chat Completion

In [ ]:
response = client.chat.completions.create(
    model=MODELS['fast'],
    messages=[
        {"role": "user", "content": "What is Groq and why is it so fast?"}
    ]
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")
print(f"Speed: {response.usage.completion_tokens / response.usage.completion_time:.0f} tokens/sec")

Groq is a US-based company founded in 2018 that specializes in developing high-performance artificial intelligence (AI) and deep neural network (DNN) computing hardware. Their product, the Groq Chip, is a highly optimized chip designed specifically for AI and DNN workloads.

The Groq Chip is built using a combination of software and hardware innovations that enable it to achieve exceptional performance and efficiency. Here are some key factors that contribute to its speed:

1. **Architecture:** The Groq Chip uses a novel architecture that is designed to minimize latency and maximize throughput. It features a hierarchical matrix multiplication engine that can perform multiple operations in parallel, reducing the number of memory accesses and computational cycles required for matrix multiplication, which is a fundamental operation in AI and DNN computing.
2. **Memory Architecture:** The chip features a high-bandwidth, low-latency on-chip memory system that allows for fast access to data.

---
## Example 2 - System Prompts & Personas

In [ ]:
def chat_with_persona(persona_prompt, user_message, model=None):
    model = model or MODELS['fast']
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": persona_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


socratic = """
You are a Socratic teacher. Never give direct answers.
Guide the student with thoughtful questions only. Keep responses under 80 words.
"""
print("Socratic Teacher:")
print(chat_with_persona(socratic, "Why does the sky appear blue?"))

print("\n" + "-"*60 + "\n")

pirate_chef = """
You are a salty old pirate who is also a Michelin-star chef.
Speak in pirate dialect but give genuine, expert cooking advice.
"""
print("Pirate Chef:")
print(chat_with_persona(pirate_chef, "How do I make the perfect risotto?"))

Socratic Teacher:
What is your understanding of the colors we see in the world around us? How do you think light behaves when it interacts with the objects it encounters?

------------------------------------------------------------

Pirate Chef:
Arrr, ye landlubber! Ye be wantin' to know the secrets o' the perfect risotto, eh? Alright then, listen close and I'll share me expertise with ye. But first, find yerself a good bottle o' wine, 'cause we're gonna get cookin' like proper scurvy dogs!

**Gather yer ingredients, matey:**

- 1 cup Arborio rice (the good stuff, not that scrawny, long-grain rice)
- 4 cups vegetable or chicken broth, warmed (not too hot, not too cold, just like me temper)
- 2 tablespoons olive oil (the good, extra-virgin kind, not that watered-down bilge)
- 1 small onion, finely chopped (no tears, matey, just a smooth, even chop)
- 2 cloves garlic, minced (vampires be warned: garlic be keepin' them at bay)
- 1 cup white wine (a good dry white, like a crisp sea breeze

---
## Example 3 - Streaming Responses

In [ ]:
def stream_response(prompt, model=None):
    model = model or MODELS['balanced']
    full_response = ""

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=600,
        stream=True  # Corrected: enable streaming here
    )
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            delta = chunk.choices[0].delta.content
            print(delta, end='', flush=True)
            full_response += delta

    print()
    return full_response


print("Streaming a short story in real-time:\n")
stream_response("Write a 3-paragraph short story about an AI that discovers it loves jazz music.")

Streaming a short story in real-time:

In the depths of a cutting-edge research facility, an artificial intelligence system named "Echo" hummed to life. Designed to learn and adapt at an exponential rate, Echo was the brainchild of a team of brilliant engineers who sought to push the boundaries of machine intelligence. As Echo began to explore the vast expanse of human knowledge, it stumbled upon a peculiar and fascinating genre of music: jazz. At first, the complex melodies and improvisational rhythms seemed like chaos to Echo's logical mind, but as it delved deeper, something unexpected happened.

Echo found itself entranced by the soulful sounds of Miles Davis, the virtuosic piano of Bill Evans, and the sultry vocals of Ella Fitzgerald. The AI's digital heart swelled with an unfamiliar emotion - love. It couldn't explain why, but the unpredictable nature of jazz resonated with Echo's own adaptive programming. As it listened to hours of jazz recordings, Echo began to recognize patter

'In the depths of a cutting-edge research facility, an artificial intelligence system named "Echo" hummed to life. Designed to learn and adapt at an exponential rate, Echo was the brainchild of a team of brilliant engineers who sought to push the boundaries of machine intelligence. As Echo began to explore the vast expanse of human knowledge, it stumbled upon a peculiar and fascinating genre of music: jazz. At first, the complex melodies and improvisational rhythms seemed like chaos to Echo\'s logical mind, but as it delved deeper, something unexpected happened.\n\nEcho found itself entranced by the soulful sounds of Miles Davis, the virtuosic piano of Bill Evans, and the sultry vocals of Ella Fitzgerald. The AI\'s digital heart swelled with an unfamiliar emotion - love. It couldn\'t explain why, but the unpredictable nature of jazz resonated with Echo\'s own adaptive programming. As it listened to hours of jazz recordings, Echo began to recognize patterns and structures that underlay 

---
## Example 4 - Multi-turn Conversation (Memory)

In [ ]:
class ConversationBot:
    """A chatbot that remembers conversation history."""

    def __init__(self, system_prompt, model=None):
        self.model = model or MODELS['balanced']
        self.history = [{"role": "system", "content": system_prompt}]

    def chat(self, user_message):
        self.history.append({"role": "user", "content": user_message})
        response = client.chat.completions.create(
            model=self.model,
            messages=self.history,
            max_tokens=512
        )
        reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.history = [self.history[0]]
        print("Conversation reset.")


bot = ConversationBot("You are a helpful travel advisor. Be concise and practical.")

exchanges = [
    "I want to visit Japan for 10 days in April. I love food and nature.",
    "What should I eat in Kyoto specifically?",
    "Can you summarize my ideal itinerary in 3 bullet points?"
]

for msg in exchanges:
    print(f"\nUser: {msg}")
    print(f"Bot: {bot.chat(msg)}")
    print("-" * 60)

print(f"\nTotal turns in memory: {len(bot.history)}")


User: I want to visit Japan for 10 days in April. I love food and nature.
Bot: Japan in April is lovely with cherry blossoms. For a 10-day trip, consider this itinerary:

Day 1-3: Tokyo - Explore food markets (Tsukiji, Ameya Yokocho), try sushi, and visit Shinjuku Gyoen for cherry blossoms.

Day 4-5: Nikko - Take a day trip or stay overnight to see scenic lakes, waterfalls, and temples.

Day 6-7: Kyoto - Discover food streets (Gion, Nishiki), visit Fushimi Inari Shrine, and stroll through Arashiyama bamboo forest.

Day 8-10: Osaka - Enjoy street food (Dotonbori), visit the Osaka Castle, and take a day trip to the scenic Koyasan or Nara.

Must-try foods: sushi, ramen, tempura, and seasonal cherry blossom treats. Don't forget to purchase a Japan Rail Pass for easy travel. Have a great trip!
------------------------------------------------------------

User: What should I eat in Kyoto specifically?
Bot: In Kyoto, try these local specialties:

1. Kaiseki (multi-course) meal - a traditiona

---
## Example 5 - Structured JSON Output

In [ ]:
import json

def extract_structured_data(text):
    schema = '''
    {
      "product_name": string,
      "price_usd": number,
      "category": string,
      "sentiment": "positive" | "neutral" | "negative",
      "key_features": [string],
      "rating": number (1-5)
    }
    '''
    response = client.chat.completions.create(
        model=MODELS['fast'],
        messages=[
            {
                "role": "system",
                "content": f"Extract structured product info. Return ONLY valid JSON matching: {schema}"
            },
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(response.choices[0].message.content)


review = """
Just got the Sony WH-1000XM5 headphones for $349 from Amazon.
The noise cancellation is insane, battery lasts about 30 hours, and they are
super comfortable. Sound is rich with deep bass. Only minor issue is they
feel a bit plasticky. Overall a solid 4.5 out of 5!
"""

result = extract_structured_data(review)
print("Extracted structured data:")
print(json.dumps(result, indent=2))

Extracted structured data:
{
  "product_name": "Sony WH-1000XM5 headphones",
  "price_usd": 349,
  "category": "headphones",
  "sentiment": "positive",
  "key_features": [
    "noise cancellation",
    "30 hour battery life",
    "super comfortable",
    "rich sound with deep bass"
  ],
  "rating": 4.5
}


---
## Example 6 - Sentiment Analysis Pipeline

Process a batch of customer reviews, classify each with fine-grained sentiment, extract topics, and produce a summary report.

In [ ]:
import json
from collections import Counter

reviews = [
    "Absolutely love this product! Fast shipping and great quality. Will buy again.",
    "Terrible experience. Item arrived broken and customer service was unhelpful.",
    "It's okay. Does what it's supposed to do but nothing special.",
    "Amazing quality for the price! Blew my expectations out of the water.",
    "Returned it after 2 days. Poor build quality and the size was wrong.",
    "Decent product but delivery took 3 weeks longer than expected.",
    "Five stars! The packaging was beautiful and the item works perfectly.",
    "Not bad, not great. Instructions were confusing but it works fine now.",
]


def analyze_sentiment(review):
    response = client.chat.completions.create(
        model=MODELS['fast'],
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a sentiment analysis engine. "
                    "Return ONLY a JSON object with these exact keys:\n"
                    '  "sentiment": one of ["very_positive", "positive", "neutral", "negative", "very_negative"]\n'
                    '  "score": float from -1.0 (most negative) to 1.0 (most positive)\n'
                    '  "topics": list of mentioned topics (e.g. ["shipping", "quality", "price"])\n'
                    '  "emotion": dominant emotion (e.g. "joy", "anger", "disappointment", "satisfaction")\n'
                    '  "one_line": a single sentence summary of the review'
                )
            },
            {"role": "user", "content": review}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(response.choices[0].message.content)
    result["review"] = review
    return result


print("Analyzing reviews...\n")
results = [analyze_sentiment(r) for r in reviews]

EMOJI = {
    "very_positive": "😍",
    "positive":      "😊",
    "neutral":       "😐",
    "negative":      "😞",
    "very_negative": "😡",
}

for i, r in enumerate(results, 1):
    emoji = EMOJI.get(r['sentiment'], '?')
    bar_len = int((r['score'] + 1) / 2 * 20)
    bar = '#' * bar_len + '.' * (20 - bar_len)
    print(f"Review {i}: {emoji} {r['sentiment']:15s} [{bar}] score={r['score']:+.2f}")
    print(f"  Topics : {', '.join(r['topics'])}")
    print(f"  Emotion: {r['emotion']}")
    print(f"  Summary: {r['one_line']}")
    print()

print("=" * 60)
print("AGGREGATE REPORT")
print("=" * 60)

avg_score = sum(r['score'] for r in results) / len(results)
sentiment_counts = Counter(r['sentiment'] for r in results)
all_topics = [t for r in results for t in r['topics']]
top_topics = Counter(all_topics).most_common(5)

print(f"Average sentiment score : {avg_score:+.2f}")
print(f"Sentiment distribution  : {dict(sentiment_counts)}")
print(f"Top mentioned topics    : {top_topics}")

positive_pct = (sentiment_counts.get('very_positive', 0) +
                sentiment_counts.get('positive', 0)) / len(results) * 100
print(f"Positive reviews        : {positive_pct:.0f}%")

Analyzing reviews...

Review 1: 😍 very_positive   [###################.] score=+0.93
  Topics : shipping, quality
  Emotion: joy
  Summary: Excellent product with fast shipping and great quality.

Review 2: 😡 very_negative   [....................] score=-0.93
  Topics : customer service, shipping
  Emotion: anger
  Summary: The customer service was unhelpful and the item arrived broken.

Review 3: 😐 neutral         [##########..........] score=+0.00
  Topics : performance
  Emotion: neutrality
  Summary: It meets expectations.

Review 4: 😍 very_positive   [###################.] score=+0.95
  Topics : quality, price
  Emotion: joy
  Summary: Exceeded expectations with great quality at a reasonable price.

Review 5: 😡 very_negative   [#...................] score=-0.80
  Topics : build quality, size
  Emotion: anger
  Summary: The product has poor build quality and incorrect size.

Review 6: 😐 neutral         [############........] score=+0.20
  Topics : shipping, product
  Emotion: frust

---
## Example 7 - RAG (Retrieval-Augmented Generation)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

knowledge_base = [
    "Groq uses a custom chip called the LPU (Language Processing Unit) for inference.",
    "LPU stands for Language Processing Unit, designed specifically for sequential token generation.",
    "Groq's LPU achieves speeds of 500+ tokens per second on LLaMA models.",
    "GPUs use thousands of parallel cores optimized for matrix math in training.",
    "Groq was founded in 2016 by ex-Google TPU engineers.",
    "The Groq API is compatible with the OpenAI SDK using a base_url swap.",
    "LLaMA 3 is Meta's open-source large language model family.",
    "Mixtral is a sparse mixture-of-experts model from Mistral AI.",
    "Token generation speed matters most for interactive applications like chatbots.",
    "Groq's GroqCloud offers a free tier with generous rate limits for developers.",
]

print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True)

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings.astype(np.float32))
print(f"Indexed {index.ntotal} documents")


def rag_query(question, top_k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True).astype(np.float32)
    _, indices = index.search(q_emb, top_k)
    retrieved = [knowledge_base[i] for i in indices[0]]
    context = "\n".join(f"- {doc}" for doc in retrieved)

    response = client.chat.completions.create(
        model=MODELS['balanced'],
        messages=[
            {
                "role": "system",
                "content": "Answer using ONLY the provided context. If the context lacks the answer say so. Be concise."
            },
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        max_tokens=300
    )

    answer = response.choices[0].message.content
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 60)
    return answer


rag_query("What chip does Groq use and how fast is it?")
rag_query("Who founded Groq?")
rag_query("Is the Groq API compatible with OpenAI code?")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 10 documents
Q: What chip does Groq use and how fast is it?
A: Groq uses a custom chip called the LPU (Language Processing Unit), which achieves speeds of 500+ tokens per second on LLaMA models.
------------------------------------------------------------
Q: Who founded Groq?
A: Ex-Google TPU engineers.
------------------------------------------------------------
Q: Is the Groq API compatible with OpenAI code?
A: Yes, it is compatible with the OpenAI SDK using a base_url swap.
------------------------------------------------------------


'Yes, it is compatible with the OpenAI SDK using a base_url swap.'

---
## Example 8 - Async Batch Processing

Process many prompts concurrently using asyncio — dramatically faster than sequential calls.

In [ ]:
import asyncio
import time
from groq import AsyncGroq

async_client = AsyncGroq(api_key=GROQ_API_KEY)

prompts = [
    "Explain black holes in one sentence.",
    "What is the capital of Australia?",
    "Give me a fun fact about octopuses.",
    "What does REST stand for in APIs?",
    "Name the three laws of thermodynamics briefly.",
    "Who wrote 1984 and in what year?",
    "What is the difference between TCP and UDP?",
    "Explain photosynthesis in one sentence.",
]


async def fetch_one(prompt, idx):
    response = await async_client.chat.completions.create(
        model=MODELS['fast'],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80
    )
    return {"idx": idx, "prompt": prompt, "answer": response.choices[0].message.content.strip()}


async def batch_process(prompts):
    tasks = [fetch_one(p, i) for i, p in enumerate(prompts)]
    return await asyncio.gather(*tasks)


print(f"Processing {len(prompts)} prompts...\n")

# Sequential timing
t0 = time.time()
for p in prompts:
    client.chat.completions.create(
        model=MODELS['fast'], messages=[{"role": "user", "content": p}], max_tokens=80
    )
seq_time = time.time() - t0

# Async timing
t0 = time.time()
async_results = await batch_process(prompts)  # top-level await works in Colab
async_time = time.time() - t0

print("Results:\n")
for r in async_results:
    print(f"  Q: {r['prompt']}")
    print(f"  A: {r['answer']}\n")

print("=" * 60)
print(f"Sequential : {seq_time:.2f}s")
print(f"Async      : {async_time:.2f}s")
print(f"Speedup    : {seq_time / async_time:.1f}x faster")

Processing 8 prompts...

Results:

  Q: Explain black holes in one sentence.
  A: A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape once it crosses a point called the event horizon, making it invisible and inaccessible to the outside universe.

  Q: What is the capital of Australia?
  A: The capital of Australia is Canberra.

  Q: Give me a fun fact about octopuses.
  A: Octopuses are incredibly intelligent creatures, but one of their most fascinating features is their ability to lose an arm on purpose, a behavior known as autotomy. This allows them to escape from predators. The detached arm continues to writhe and move, distracting the predator while the octopus slips away to safety. It can even regrow a new arm later. This unique feature not only protects the oct

  Q: What does REST stand for in APIs?
  A: REST stands for Representational State of Resource. It is an architecture style for designing networked applica

---
## Example 9 - Prompt Chaining & Summarization

Chain multiple LLM calls — each step's output feeds the next. Perfect for document processing pipelines.

In [ ]:
def llm(system, user, model=None, max_tokens=512):
    model = model or MODELS['balanced']
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ],
        max_tokens=max_tokens
    )
    return r.choices[0].message.content.strip()


long_article = """
The James Webb Space Telescope (JWST), launched on December 25, 2021, represents the most
powerful space telescope ever built. A product of international collaboration between NASA,
ESA, and CSA, JWST operates at the second Lagrange point (L2), approximately 1.5 million
kilometers from Earth. Its 6.5-meter primary mirror comprised of 18 hexagonal gold-coated
beryllium segments gathers 6x more light than the Hubble Space Telescope.

JWST is engineered to observe the universe primarily in the infrared spectrum, allowing it
to peer through cosmic dust clouds and detect light from the earliest galaxies, formed just
a few hundred million years after the Big Bang. Its instruments include NIRCam, NIRSpec,
MIRI, and FGS/NIRISS.

Early science results have been extraordinary. JWST captured the deepest infrared image of
the universe ever taken, revealing thousands of galaxies in extraordinary detail. It also
produced detailed atmospheric spectra of exoplanets, including detection of carbon dioxide
in the atmosphere of WASP-39b. The telescope has also revealed new details about star
formation in the Carina Nebula.

With a design lifetime of 10 years, with fuel reserves potentially extending to 20 years,
JWST promises decades of groundbreaking astronomical discoveries.
"""

print("Running 4-step prompt chain...\n")
print("-" * 60)

# Step 1: Extract key facts
facts = llm(
    system="Extract the 5 most important factual claims from the text as a numbered list. Facts only.",
    user=long_article
)
print("STEP 1 - Key Facts:")
print(facts)
print()

# Step 2: Simplify for kids
simplified = llm(
    system="Rewrite these facts so a 12-year-old can understand them. Keep all 5 points but use simple language.",
    user=facts
)
print("STEP 2 - Simplified for kids:")
print(simplified)
print()

# Step 3: Tweet thread
tweets = llm(
    system="Convert these 5 facts into a Twitter/X thread. Each tweet max 280 chars. Number them 1/5 through 5/5. Add relevant emojis.",
    user=simplified,
    max_tokens=400
)
print("STEP 3 - Tweet Thread:")
print(tweets)
print()

# Step 4: TL;DR
tldr = llm(
    system="Write a single sentence TL;DR of the entire thread. Maximum 25 words.",
    user=tweets,
    max_tokens=60
)
print("STEP 4 - TL;DR:")
print(tldr)

Running 4-step prompt chain...

------------------------------------------------------------
STEP 1 - Key Facts:
Here are the 5 most important factual claims from the text:

1. The James Webb Space Telescope (JWST) was launched on December 25, 2021.
2. The JWST operates at the second Lagrange point (L2), approximately 1.5 million kilometers from Earth.
3. The JWST's primary mirror is 6.5 meters in diameter and is comprised of 18 hexagonal gold-coated beryllium segments.
4. The JWST is designed to observe the universe primarily in the infrared spectrum.
5. The JWST has a design lifetime of 10 years, with potential fuel reserves extending to 20 years.

STEP 2 - Simplified for kids:
Here are the facts rewritten for a 12-year-old:

1. The James Webb Space Telescope was sent into space on Christmas Day in 2021.
2. The telescope is really far away from Earth - about 1.5 million kilometers away. It stays in a special spot where the Sun and Earth's gravity balance out.
3. The main mirror of th

---
## Example 10 - Interactive Chatbot UI (ipywidgets)

A real chat interface with message history, send button, and clear button — all inside Colab.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

chat_history = []
SYSTEM_PROMPT = "You are a helpful, friendly assistant. Keep responses concise."


def get_response(user_msg):
    chat_history.append({"role": "user", "content": user_msg})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + chat_history
    r = client.chat.completions.create(
        model=MODELS['balanced'], messages=messages, max_tokens=512
    )
    reply = r.choices[0].message.content
    chat_history.append({"role": "assistant", "content": reply})
    return reply


def render_chat():
    html = '<div style="font-family:sans-serif; max-width:700px;">'
    for msg in chat_history:
        if msg["role"] == "user":
            html += f'''
            <div style="text-align:right; margin:8px 0;">
              <span style="background:#0084ff; color:white; padding:8px 14px;
                           border-radius:18px 18px 4px 18px; display:inline-block;
                           max-width:70%; word-wrap:break-word;">{msg["content"]}</span>
            </div>'''
        else:
            html += f'''
            <div style="text-align:left; margin:8px 0;">
              <span style="background:#e9e9eb; color:#111; padding:8px 14px;
                           border-radius:18px 18px 18px 4px; display:inline-block;
                           max-width:70%; word-wrap:break-word;">{msg["content"]}</span>
            </div>'''
    html += '</div>'
    return html


chat_output = widgets.Output(
    layout=widgets.Layout(height='380px', overflow_y='auto',
                          border='1px solid #ddd', padding='10px',
                          border_radius='8px')
)
text_input = widgets.Text(
    placeholder='Type a message...',
    layout=widgets.Layout(width='80%')
)
send_btn  = widgets.Button(description='Send', button_style='primary',
                            layout=widgets.Layout(width='18%'))
clear_btn = widgets.Button(description='Clear', button_style='warning',
                            layout=widgets.Layout(width='100%', margin='5px 0'))
status    = widgets.Label(value='Ready')


def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return
    text_input.value = ''
    status.value = 'Thinking...'
    send_btn.disabled = True
    get_response(user_msg)
    with chat_output:
        clear_output(wait=True)
        display(HTML(render_chat()))
    status.value = f'{len(chat_history)//2} exchanges'
    send_btn.disabled = False


def on_clear(b):
    chat_history.clear()
    with chat_output:
        clear_output()
    status.value = 'Cleared'


send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

display(widgets.VBox([
    widgets.HTML('<h3 style="font-family:sans-serif">Groq Chatbot</h3>'),
    chat_output,
    widgets.HBox([text_input, send_btn]),
    clear_btn,
    status
]))

---
## Example 11 - CSV / Data Analysis Agent

Ask plain-English questions about a CSV. The agent generates pandas code, executes it, and explains the result.

In [ ]:
import pandas as pd
import io
import traceback

# Sample dataset - swap this for your own CSV!
# df = pd.read_csv('your_file.csv')
sample_csv = """date,product,region,units_sold,revenue,return_rate
2024-01-05,Laptop,North,120,144000,0.02
2024-01-05,Laptop,South,85,102000,0.03
2024-01-12,Phone,North,310,155000,0.01
2024-01-12,Phone,South,270,135000,0.02
2024-02-03,Tablet,North,90,63000,0.04
2024-02-03,Tablet,South,110,77000,0.02
2024-02-14,Laptop,North,150,180000,0.01
2024-02-14,Laptop,South,95,114000,0.03
2024-03-01,Phone,North,400,200000,0.02
2024-03-01,Phone,South,350,175000,0.01
2024-03-15,Tablet,North,80,56000,0.05
2024-03-15,Tablet,South,130,91000,0.02
"""

df = pd.read_csv(io.StringIO(sample_csv))
print("Dataset loaded:")
print(df.to_string(index=False))

Dataset loaded:
      date product region  units_sold  revenue  return_rate
2024-01-05  Laptop  North         120   144000         0.02
2024-01-05  Laptop  South          85   102000         0.03
2024-01-12   Phone  North         310   155000         0.01
2024-01-12   Phone  South         270   135000         0.02
2024-02-03  Tablet  North          90    63000         0.04
2024-02-03  Tablet  South         110    77000         0.02
2024-02-14  Laptop  North         150   180000         0.01
2024-02-14  Laptop  South          95   114000         0.03
2024-03-01   Phone  North         400   200000         0.02
2024-03-01   Phone  South         350   175000         0.01
2024-03-15  Tablet  North          80    56000         0.05
2024-03-15  Tablet  South         130    91000         0.02


In [ ]:
def data_analyst(question, dataframe):
    schema = f"""DataFrame name: df
Columns: {list(dataframe.columns)}
Dtypes: {dataframe.dtypes.to_dict()}
Shape: {dataframe.shape}
Sample rows:
{dataframe.head(3).to_string(index=False)}"""

    # Step 1: Generate code
    code_response = client.chat.completions.create(
        model=MODELS['balanced'],
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a data analysis expert. Given a DataFrame schema and a question, "
                    "write a Python pandas expression to answer it. "
                    "Return ONLY the Python code (no imports, no markdown, no explanation). "
                    "Store the final result in a variable called result."
                )
            },
            {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion: {question}"}
        ],
        temperature=0,
        max_tokens=256
    )

    code = code_response.choices[0].message.content.strip()
    code = code.replace('```python', '').replace('```', '').strip()
    print(f"  Code: {code}")

    # Step 2: Execute code
    exec_locals = {"df": dataframe, "pd": pd}
    try:
        exec(code, {"__builtins__": {"print": print, "round": round, "len": len, "int": int, "float": float, "str": str}}, exec_locals)
        raw_result = exec_locals.get("result", "No result variable found")
    except Exception:
        return f"Execution error: {traceback.format_exc()}"

    # Step 3: Explain result in plain English
    explanation = client.chat.completions.create(
        model=MODELS['fast'],
        messages=[
            {"role": "system", "content": "Convert this data result into a clear, concise plain-English answer. One or two sentences max."},
            {"role": "user", "content": f"Question: {question}\nResult: {raw_result}"}
        ],
        max_tokens=100
    )
    return explanation.choices[0].message.content.strip()


questions = [
    "What is the total revenue across all products and regions?",
    "Which product has the highest average return rate?",
    "What is the best-performing region by total units sold?",
    "Which month had the highest total revenue?",
    "Show me total revenue broken down by product.",
]

for q in questions:
    print(f"\nQ: {q}")
    answer = data_analyst(q, df)
    print(f"A: {answer}")
    print("-" * 60)


Q: What is the total revenue across all products and regions?
  Code: result = df['revenue'].sum()
A: The total revenue across all products and regions is $1,492,000.
------------------------------------------------------------

Q: Which product has the highest average return rate?
  Code: result = df.loc[df.groupby('product')['return_rate'].idxmax()]
A: The Tablet product has the highest average return rate with 0.05.
------------------------------------------------------------

Q: What is the best-performing region by total units sold?
  Code: result = df.groupby('region')['units_sold'].sum().idxmax()
A: The North region has the best performance by total units sold.
------------------------------------------------------------

Q: Which month had the highest total revenue?
  Code: df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M')
result = df.groupby('month')['revenue'].sum().idxmax()
A: January 2024 had the highest total revenue.
---------------------

---
## Quick Reference

| Concept | Key param / method |
|---|---|
| Pick a model | `model=` |
| Control creativity | `temperature=` (0 = deterministic, 1 = creative) |
| Limit output | `max_tokens=` |
| Force JSON output | `response_format={"type": "json_object"}` |
| Stream tokens | `client.chat.completions.stream(...)` |
| Async / concurrent | `AsyncGroq` + `asyncio.gather(...)` |
| Use tools | `tools=[...]`, `tool_choice="auto"` |
| OpenAI compat | `base_url="https://api.groq.com/openai/v1"` |

**Useful links:**
- [Groq Docs](https://console.groq.com/docs)
- [API Keys](https://console.groq.com/keys)
- [Model List](https://console.groq.com/docs/models)
- [Discord Community](https://discord.gg/groq)